# 💊 YOLO26n — Medical Pill Counting (Google Colab)

**Goal:** train a single-class detector that finds pills in a blister pack image and reports the pill count.

**Steps:**
1. Upload your zipped dataset to Google Drive
2. Run all cells in order
3. Download `best.pt` at the end

> ⚡ **Runtime → Change runtime type → T4 GPU** (free tier)

In [ ]:
# Cell 1: Build a one-class dataset for pill counting
from pathlib import Path
import shutil

# Keep only pill annotations and drop the empty-slot class.
SOURCE_ROOT = Path('/content/dataset/archive')
TARGET_ROOT = Path('/content/dataset_pillcount/archive')

for split in ['train', 'valid']:
    src_images = SOURCE_ROOT / split / 'images'
    src_labels = SOURCE_ROOT / split / 'labels'
    dst_images = TARGET_ROOT / split / 'images'
    dst_labels = TARGET_ROOT / split / 'labels'
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    for image_path in src_images.glob('*'):
        shutil.copy2(image_path, dst_images / image_path.name)

    for label_path in src_labels.glob('*.txt'):
        filtered_lines = []
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            if parts[0] == '0':
                filtered_lines.append('0 ' + ' '.join(parts[1:]))
        (dst_labels / label_path.name).write_text('\n'.join(filtered_lines) + ('\n' if filtered_lines else ''))

print('Prepared one-class pill-count dataset at:', TARGET_ROOT)

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────
!nvidia-smi

Sat Apr 25 08:36:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# ── Cell 2: Install Ultralytics (supports YOLO26) ──────────────
!pip install ultralytics --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:00


In [2]:
# ── Cell 3: Mount Google Drive ────────────────────────────────
# Upload your zipped archive folder to Google Drive first!
# File to upload: E:/FOS/ocr/archive.zip  (zip the whole 'archive' folder)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ── Cell 4: Unzip dataset ─────────────────────────────────────
import os

# ✏️ CHANGE THIS to wherever you placed archive.zip in your Drive
ZIP_PATH = '/content/drive/MyDrive/archive_upload.zip'

!unzip -q "{ZIP_PATH}" -d /content/dataset
!ls /content/dataset

archive


In [ ]:
# ── Cell 2: Write the one-class dataset YAML ───────────────────
# This points Ultralytics at the filtered dataset and trains only the pill class.
yaml_content = """
path: /content/dataset/archive
train: train/images
val:   valid/images

names:
  0: pill
"""

with open('/content/pills.yaml', 'w') as f:
    f.write(yaml_content)

print('YAML written:')
!cat /content/pills.yaml

YAML written:

path: /content/dataset/archive
train: train/images
val:   valid/images

names:
  0: occupied
  1: vacant


In [7]:
# ── Cell 3: Verify the filtered dataset structure ──────────────
import pathlib

base = pathlib.Path('/content/dataset_pillcount/archive')
for split in ['train', 'valid']:
    imgs = list((base / split / 'images').glob('*'))
    lbls = list((base / split / 'labels').glob('*.txt'))
    print(f'{split}: {len(imgs)} images, {len(lbls)} labels')

train: 224 images, 223 labels
valid: 46 images, 46 labels


In [1]:
# ── Cell 4: Train YOLO26n for pill counting ───────────────────
from ultralytics import YOLO

# This trains a single-class detector named 'pill'.
# The model learns to detect each pill, then we count detections.
model = YOLO('yolo26n.pt')

results = model.train(
    data='/content/pills.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project='/content/runs',
    name='pill_count_yolo26',
    exist_ok=True,
)

print('\n✅ Training complete!')

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
# ── Cell 5: Show results ──────────────────────────────────────
from IPython.display import Image, display
import glob

# Show training curves
for img_path in glob.glob('/content/runs/pill_count_yolo26/*.png'):
    print(img_path)
    display(Image(img_path))

In [ ]:
# ── Cell 6: Copy best.pt to Google Drive ─────────────────────
import shutil

src = '/content/runs/pill_count_yolo26/weights/best.pt'
dst = '/content/drive/MyDrive/pill_count_yolo26_best.pt'

shutil.copy(src, dst)
print(f'✅  best.pt saved to Google Drive → {dst}')
print('Copy it back to your workspace and use it for pill counting.')

In [ ]:
# ── Cell 7 (Optional): Count pills in a sample image ─────────
# Upload a test image first (Files panel on left) or use a Drive path
TEST_IMAGE = '/content/drive/MyDrive/test_pill.jpg'   # change this

from PIL import Image as PILImage

model_inf = YOLO('/content/runs/pill_count_yolo26/weights/best.pt')
results = model_inf(TEST_IMAGE, conf=0.25)

for r in results:
    pill_count = len(r.boxes)
    print(f'Pill count: {pill_count}')
    annotated = r.plot()
    display(PILImage.fromarray(annotated[..., ::-1]))

In [1]:
from ultralytics import YOLO

model = YOLO(r"E:\FOS\ocr\pill_count_yolo26_best.pt")  # adjust path
results = model.predict(source=r"E:\FOS\ocr\image.jpg", conf=0.25, save=True)

for r in results:
    pill_count = len(r.boxes)
    print("Pill count:", pill_count)

FileNotFoundError: E:\FOS\ocr\image.jpg does not exist

In [ ]:
# ── Cell 8 (Optional): Pill count + medicine name + quantity ──
!apt-get update -qq
!apt-get install -y tesseract-ocr -qq

import re
import sys
import cv2
import pytesseract
from PIL import Image as PILImage
from IPython.display import display
from ultralytics import YOLO

if sys.platform.startswith("win"):
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

IMAGE_PATH = '/content/drive/MyDrive/medicine.jpg'   # change this to your image
MODEL_PATH = '/content/runs/pill_count_yolo26/weights/best.pt'

OCR_STOP_WORDS = {
    'all',
    'age',
    'bottle',
    'capsule',
    'capsules',
    'enjoy',
    'for',
    'group',
    'herbs',
    'mg',
    'ml',
    'oral',
    'pill',
    'suspension',
    'syrup',
    'tablets',
    'tonic',
    'with',
}
OCR_CONFIDENCE_THRESHOLD = 35


def rotate_image(image, angle):
    if angle == 90:
        return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    if angle == 180:
        return cv2.rotate(image, cv2.ROTATE_180)
    if angle == 270:
        return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
    return image


def preprocess_for_ocr(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]


def collect_ocr_tokens(image_bgr, config='--oem 3 --psm 11'):
    data = pytesseract.image_to_data(
        image_bgr,
        config=config,
        output_type=pytesseract.Output.DICT,
    )

    tokens = []
    for text, conf in zip(data['text'], data['conf']):
        text = text.strip()
        if not text:
            continue
        try:
            conf_value = float(conf)
        except ValueError:
            conf_value = -1
        tokens.append((text, conf_value))
    return tokens


def ocr_variants(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    height = gray.shape[0]
    crops = [gray[: max(int(height * 0.20), 1), :], gray]
    variants = []
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

    for crop in crops:
        variants.append(crop)
        variants.append(cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC))
        variants.append(cv2.GaussianBlur(crop, (3, 3), 0))
        variants.append(cv2.morphologyEx(crop, cv2.MORPH_CLOSE, kernel))
        _, otsu = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        variants.append(otsu)
        adaptive = cv2.adaptiveThreshold(
            crop,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            31,
            11,
        )
        variants.append(adaptive)

    texts = []
    seen = set()
    for variant in variants:
        for angle in [0, 90, 180, 270]:
            rotated = rotate_image(variant, angle)
            for psm in [6, 11, 7]:
                confident_tokens = [token for token, conf in collect_ocr_tokens(rotated, config=f'--oem 3 --psm {psm}') if conf >= OCR_CONFIDENCE_THRESHOLD]
                confident_text = ' '.join(confident_tokens).strip()
                if confident_text and confident_text not in seen:
                    seen.add(confident_text)
                    texts.append(confident_text)

                raw_text = pytesseract.image_to_string(rotated, config=f'--oem 3 --psm {psm}').strip()
                if raw_text and raw_text not in seen:
                    seen.add(raw_text)
                    texts.append(raw_text)
    return texts


def best_ocr_text(image_bgr):
    texts = ocr_variants(image_bgr)
    return '\n'.join(texts)


def score_candidate(line, frequency):
    words = re.findall(r"[A-Za-z][A-Za-z'&+-]*", line)
    if not words:
        return -1

    lowered_words = [word.lower() for word in words]
    if all(word in OCR_STOP_WORDS for word in lowered_words):
        return -1

    alpha_count = sum(1 for char in line if char.isalpha())
    digit_count = sum(1 for char in line if char.isdigit())
    stopword_count = sum(1 for word in lowered_words if word in OCR_STOP_WORDS)
    score = alpha_count * 2 + frequency * 6 + len(words) * 3 - digit_count * 2 - stopword_count * 4

    if 1 <= len(words) <= 4:
        score += 5
    if any(word not in OCR_STOP_WORDS and len(word) > 2 for word in lowered_words):
        score += 4
    if any(char.isupper() for char in line):
        score += 2

    return score


def parse_name_and_quantity(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    quantity = None
    quantity_match = re.search(
        r'\b\d+(?:\.\d+)?\s*(?:ml|mg|mcg|g|l|liters?|tablets?|capsules?|tabs?|syrup|tonic)\b',
        text,
        flags=re.I,
    )
    if quantity_match:
        quantity = quantity_match.group(0)

    line_frequency = {}
    normalized_lines = []
    for line in lines:
        normalized = re.sub(r'\s+', ' ', line).strip()
        if not normalized:
            continue
        normalized_lines.append(normalized)
        key = normalized.lower()
        line_frequency[key] = line_frequency.get(key, 0) + 1

    candidates = []
    for line in normalized_lines:
        if quantity and quantity.lower() in line.lower():
            continue
        if re.search(r'\b\d+\s*(?:ml|mg|mcg|g|l|liters?|tablets?|capsules?|tabs?|syrup|tonic)\b', line, flags=re.I):
            continue

        cleaned = re.sub(r'[^A-Za-z0-9+&\- ]', ' ', line).strip()
        cleaned = re.sub(r'\s+', ' ', cleaned)
        if len(cleaned) < 3:
            continue

        if not any(char.isalpha() for char in cleaned):
            continue

        score = score_candidate(cleaned, line_frequency.get(cleaned.lower(), 1))
        if score >= 0:
            candidates.append((score, cleaned))

    medicine_name = max(candidates, key=lambda item: item[0], default=(None, None))[1]
    return medicine_name, quantity

image_bgr = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise FileNotFoundError(f'Could not read image: {IMAGE_PATH}')

model = YOLO(MODEL_PATH)
results = model.predict(source=IMAGE_PATH, conf=0.25, save=False)

ocr_text = best_ocr_text(image_bgr)
medicine_name, quantity = parse_name_and_quantity(ocr_text)

for r in results:
    pill_count = len(r.boxes)
    print(f'Pill count: {pill_count}')
    print(f'Medicine name: {medicine_name or "Not found"}')
    print(f'Quantity: {quantity or "Not found"}')
    print('OCR text:')
    print(ocr_text if ocr_text.strip() else 'No text detected')
    annotated = r.plot()
    display(PILImage.fromarray(annotated[..., ::-1]))

<>:12: SyntaxWarning: invalid escape sequence '\F'
<>:13: SyntaxWarning: invalid escape sequence '\F'
<>:12: SyntaxWarning: invalid escape sequence '\F'
<>:13: SyntaxWarning: invalid escape sequence '\F'
C:\Users\hp\AppData\Local\Temp\ipykernel_22000\2066762863.py:12: SyntaxWarning: invalid escape sequence '\F'
  IMAGE_PATH = 'E:\FOS\ocr\images4.jpg'   # change this to your image
C:\Users\hp\AppData\Local\Temp\ipykernel_22000\2066762863.py:13: SyntaxWarning: invalid escape sequence '\F'
  MODEL_PATH = 'E:\FOS\ocr\pill_count_yolo26_best.pt'
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



image 1/1 E:\FOS\ocr\images4.jpg: 480x640 10 pills, 100.1ms
Speed: 38.2ms preprocess, 100.1ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)


C:\Users\hp\AppData\Local\Temp\ipykernel_22000\2066762863.py:12: SyntaxWarning: invalid escape sequence '\F'
  IMAGE_PATH = 'E:\FOS\ocr\images4.jpg'   # change this to your image
C:\Users\hp\AppData\Local\Temp\ipykernel_22000\2066762863.py:13: SyntaxWarning: invalid escape sequence '\F'
  MODEL_PATH = 'E:\FOS\ocr\pill_count_yolo26_best.pt'


TesseractNotFoundError: tesseract is not installed or it's not in your PATH. See README file for more information.